<center>
<img src="../../img/ods_stickers.jpg" />
    
## [mlcourse.ai](https://mlcourse.ai) – دورة التعلم الآلي المفتوحة 
### <center> المؤلف: ألكسندر نيتشيبورينكو، AlexNich
    
## <center> البرنامج التعليمي
### <center> "الكشف عن الحالات الشاذة: الغابة المعزولة"



<center>
<img src="../../img/out_liar.jpg" />



# مقدمة.



في الدورة التدريبية الخاصة بنا، تطرقنا إلى حد ما إلى مهام التعلم غير الخاضعة للرقابة (تقليل الأبعاد والتجميع)، وبقي فصل آخر مهم دون أن يلاحظه أحد - الكشف عن الحالات الشاذة.
في علم البيانات، كشف الشذوذ (الكشف عن الأشياء الشاذة أو الجديدة) هو تحديد العناصر أو الأحداث أو الملاحظات النادرة التي تثير الشكوك من خلال اختلافها بشكل كبير عن غالبية البيانات. عادةً ما تُترجم العناصر الشاذة إلى مشكلة ما مثل الاحتيال البنكي أو عيب هيكلي أو مشاكل طبية أو أخطاء في النص. ويشار إلى الحالات الشاذة أيضًا على أنها القيم المتطرفة والمستجدات والضوضاء والانحرافات والاستثناءات. غالبًا ما تقلل القيم المتطرفة من جودة خوارزميات تعلم الآلة لأن النماذج تتناغم معها.



# القليل من النظرية والفكرة الرئيسية للخوارزمية.


إحدى خوارزميات الكشف عن الشذوذ التي أثبتت جدواها هي Isolation Forest. كما يوحي الاسم - هذه مجموعة من الأشجار التي تم بناؤها بشكل مستقل عن بعضها البعض. لكن في هذه الحالة يختلف مبدأ بناء الشجرة عما يستخدم في مشاكل الانحدار أو التصنيف - وهو تقليل معيار التقسيم في كل خطوة.
الأشجار أيضًا ثنائية، ولكن في كل عقدة يتم اختيار الميزة بشكل عشوائي، ويتم أيضًا اختيار قيم الميزة للتقسيم بشكل عشوائي من النطاق (min، max) الذي تقبله الميزة. يتم بناء الشجرة إلى أقصى عمق ممكن - عندما يكون هناك كائن واحد فقط في كل ورقة.
مع هذا النهج، يبدو أن الحالات الشاذة ستصل إلى الورقة النهائية في وقت أبكر بكثير من الكائنات العادية. هذا هو مبدأ اكتشاف الحالات الشاذة الذي تستخدمه Isolation Forest، وتقوم هذه الخوارزمية "بعزل" الحالات الشاذة عن طريق الكائنات العادية في خطوات مبكرة.
ربما لا يبدو الأمر واضحًا تمامًا، ولكن يمكننا النظر في مثال اللعبة أحادي البعد التالي - مثل هذه المجموعة من الأرقام [1،20،21،25]. من الواضح أن الرقم المتطرف في هذه الحالة هو الرقم 1. إذا اخترنا عتبة الانقسام الأول (1,25)، ففي الغالبية العظمى من الحالات، سيتم "عزل" الرقم 1 على الفور في الورقة الأولى من الشجرة. دعونا نحاكي هذا الموقف لـ 1000 اختيار عتبة عشوائي.


In [ ]:
# Importing libaries

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# Toy example

hits = 0
K = 1000
for k in range(K):
    np.random.seed(k + 101)
    split = np.random.uniform(1, 25)
    if split < 20:
        hits += 1

print('The portion of cases when the "1" goes to the first leaf:', hits / K)

بهذه الطريقة، يمكننا قياس درجة الشذوذ باستخدام طول مسار الكائن، أي عدد الحواف التي يجب أن تمر بها الملاحظة في الشجرة بدءًا من الجذر إلى العقدة الطرفية. لكن لدينا مشكلة، بالنسبة لعينة البيانات $X$= {$x_1,...,x_n$} فإن أقصى ارتفاع ممكن لشجرة العزل ينمو بترتيب $n$، وينمو متوسط ​​طول المسار بترتيب $log({n})$. لذلك، لا يمكننا مقارنة شذوذ الكائنات في عينات ذات أحجام مختلفة، والتطبيع بأي من القيم المذكورة أعلاه لن يساعد أيضًا. لذلك سوف نستخدم هذه الصيغة للتطبيع:
## $$c{(n)} = 2H(n-1) - {2 (n-1)\over n}$$
حيث $H(n-1)$ هو $n$-$Harmonic$ الرقم:
## $$H(n-1) = \sum_{k=1}^{n-1} {1\over k} \approx \gamma\ {(Euler's\ constant)} + \ln{(n-1)} \approx 0.5772156649 + \ln{(n-1)}$$
$c({n})$ يعطي متوسط طول المسار للبحث غير الناجح في شجرة البحث الثنائية (BST). يمكننا استخدامه لأن شجرة العزل لها بنية مكافئة لـ BST و$c({n})$ تساوي تقدير المتوسط ​​$h({x})$ للعقد الخارجية.
لذلك يتم حساب درجة الشذوذ النهائية بهذه الصيغة:
## $$S(x,n) = {2 ^ {E(x)\over c(n)}}$$
حيث $E(x)$ - متوسط طول المسار في أشجار غابتنا حيث تم عزل المثال $x$:
## $$E(x) = {1\over N}\sum_{i=1}^{N} {h(x)_i}$$
و$N$ - عدد الأشجار في الغابة.
$S(x,n)$ يتغير من $0$ إلى $1$. عندما يكون $S(x,n)$ المثال قريبًا جدًا من $1$ فهذا يعني أنه بالتأكيد حالة شاذة، عندما يكون أصغر بكثير من $0.5$، فإن هذا المثال آمن ليتم اعتباره مثالًا عاديًا، وإذا كانت جميع الأمثلة تحتوي على $S(x,n) \approx 0.5$، فإن البيانات بأكملها لا تحتوي على أي شذوذ مميز.
عندما نقرر أي مثال يعتبر حالة شاذة، يمكننا اختيار جزء من الأمثلة ذات الدرجات العالية أو وضع حد في $S(x,n)$.



# دعونا ننمي غابتنا المعزولة!


في هذا الجزء من البرنامج التعليمي، سنقوم بتنفيذ غابة العزل الخاصة بنا ونرى كيف تعمل مع القيم المتطرفة والكائنات العادية.


In [ ]:
from math import log as ln

import matplotlib.pyplot as plt
# Importing libaries ----
import numpy as np
import pandas as pd
import seaborn as sns

In [ ]:
# External Node - leaf with 1 example


class ExNode:
    def __init__(self, size):
        self.size = size

In [ ]:
# Internal Node


class InNode:
    def __init__(self, left, right, split_feature, split_threshold):
        self.left = left
        self.right = right
        self.split_feature = split_feature
        self.split_threshold = split_threshold

In [ ]:
# Build one Isolation tree


def IsolationTree(X):
    if len(X) <= 1:
        return ExNode(len(X))
    else:
        f = np.random.choice(X.columns)
        t = np.random.uniform(X[f].min(), X[f].max())
        X_l = X[X[f] < t]
        X_r = X[X[f] >= t]
        return InNode(IsolationTree(X_l), IsolationTree(X_r), f, t)

In [ ]:
# Build forest


def MyIsolationForest(X, n_trees):
    forest = []
    for i in range(n_trees):
        forest.append(IsolationTree(X))
    return forest

In [ ]:
# Depth of external node where object was isolated


def path_length(x, tree, curr_depth):
    if isinstance(tree, ExNode):
        return curr_depth
    t = tree.split_feature
    if x[t] < tree.split_threshold:
        return path_length(x, tree.left, curr_depth + 1)
    else:
        return path_length(x, tree.right, curr_depth + 1)


الوظائف المطلوبة لحساب درجة الشذوذ: $E(d), H(x), c(n), S(x,n)$.


In [ ]:
def E(d):
    return np.mean(d)


def H(x):
    return ln(x) + 0.5772156649


def c(n):
    return 2 * H(n - 1) - 2 * (n - 1) / n if n > 2 else 1 if n == 1 else 0


def S(x, n):
    return 2 ** (-E(x) / c(n))


## دعونا نجد القيم المتطرفة باستخدام غابتنا!



أولاً، سنقوم بإنشاء بيانات أحادية الأبعاد - التوزيع الطبيعي وإيجاد متوسط طول المسار و$S(x,n)$ للكائنات العادية والشاذة يعتمد على عدد الأشجار.


In [ ]:
# Generating normal distributed 1d-data
random_generator = np.random.RandomState(42)

true_mean = 100
true_sigma = 10

X_all = random_generator.normal(true_mean, true_sigma, size=500)

print("Normal interval:", true_mean - 2 * true_sigma, "-", true_mean + 2 * true_sigma)

X_outliers = pd.DataFrame(
    np.hstack(
        [
            X_all[X_all < true_mean - 2 * true_sigma],
            X_all[X_all > true_mean + 2 * true_sigma],
        ]
    ),
    columns=["x"],
)
X_normal = pd.DataFrame(list(set(X_all).difference(set(X_outliers))), columns=["x"])
X_all = pd.DataFrame(X_all, columns=["x"])

print("Partition of outliers:", len(X_outliers) / len(X_all))

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(X_all["x"], bins=10)
plt.hist(X_outliers["x"], bins=10)
plt.legend(["X_all", "X_outliers"])
plt.ylabel("Count")
plt.xlabel("X value")
plt.title("Distribution of X");

In [ ]:
# Outlier for test

X_outliers.iloc[2, :]

In [ ]:
# Normal example for test

X_normal.iloc[0, :]

In [ ]:
%%time

anomaly_x = []
normal_x = []

anomaly_mean_depth = []
normal_mean_depth = []

anomaly_S = []
normal_S = []

for n in range(1, 51, 1):
    MyIF = MyIsolationForest(X_all, n)
    for iTree in MyIF:
        anomaly_x.append(path_length(X_outliers.iloc[2, :], iTree, 0))
        normal_x.append(path_length(X_normal.iloc[0, :], iTree, 0))
    anomaly_mean_depth.append(E(anomaly_x))
    normal_mean_depth.append(E(normal_x))
    anomaly_S.append(S(anomaly_x, len(X_all)))
    normal_S.append(S(normal_x, len(X_all)))

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(1, 51, 1), anomaly_mean_depth, c="r")
plt.plot(range(1, 51, 1), normal_mean_depth, c="g")
plt.title("Average path length (n_trees)")
plt.legend(["anomaly", "normal"])
plt.xlabel("Number of trees")
plt.ylabel("Average path length");

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(1, 51, 1), anomaly_S, c="r")
plt.plot(range(1, 51, 1), normal_S, c="g")
plt.title("S (n_trees)")
plt.legend(["anomaly", "normal"])
plt.xlabel("Number of trees")
plt.ylabel("S(x,n)");


كما يمكننا أن نرى، من الواضح أن الكائن الشاذ له عمق مسار أقل وأكبر $S(x,n)$. كما أننا لا نحتاج إلى العديد من الأشجار للكشف عن الحالات الشاذة، فحوالي 15 شجرة نصل إلى الخط المقارب.
حسنًا، فلنبحث عن القيم المتطرفة في البيانات ثنائية الأبعاد. سوف نستخدم IsolationForest مع 30 شجرة للعثور على ذراتنا والتحقق من جودة الكشف.


In [ ]:
# Generating 2d-data
random_generator = np.random.RandomState(42)

# Generating normal data
X_normal = random_generator.randn(2000, 2) * 0.5
X_normal = pd.DataFrame(X_normal, columns=["x1", "x2"])
X_normal["type"] = "normal"

# Generating outliers
X_outliers_1 = random_generator.uniform(low=-6, high=6, size=(78, 2))
X_outliers_2 = random_generator.uniform(low=-6, high=-3, size=(35, 2))

X_outliers = np.vstack([X_outliers_1, X_outliers_2])

X_outliers = pd.DataFrame(X_outliers, columns=["x1", "x2"])
X_outliers["R"] = X_outliers["R"] = np.sqrt(
    X_outliers["x1"] ** 2 + X_outliers["x2"] ** 2
)
X_outliers = X_outliers[X_outliers["R"] > 3].drop(columns=["R"])
X_outliers["type"] = "anomaly"

# Full data

X_full = pd.concat([X_normal, X_outliers])

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(X_outliers["x1"], X_outliers["x2"], c="r")
plt.scatter(X_normal["x1"], X_normal["x2"], c="g")
plt.xlabel("x1")
plt.ylabel("x2")
plt.legend(["outliers", "normal"])
plt.title("2d distribution");

In [ ]:
X_normal.shape, X_outliers.shape, X_full.shape

In [ ]:
%%time
MyIF = MyIsolationForest(X_full[["x1", "x2"]], 30)

In [ ]:
X_outliers.iloc[0, :]

In [ ]:
X_normal.iloc[0, :]

In [ ]:
%%time

aScore = []


for i in range(X_full.shape[0]):
    depth = []
    for iTree in MyIF:
        depth.append(path_length(X_full.iloc[i, :], iTree, 0))

    aScore.append(S(depth, X_full.shape[0]))

In [ ]:
X_full["aScore"] = aScore

In [ ]:
t = X_full["aScore"].quantile(0.95)
X_full["Outlier"] = X_full["aScore"].apply(
    lambda x: -1 if x >= t else 1
)  # -1 for outliers and 1 for normal object

In [ ]:
plt.hist(X_full["aScore"])
plt.xlabel("Anomaly Score");

In [ ]:
plt.figure(figsize=(10, 7))
plt.scatter(X_full["x1"], X_full["x2"], c=X_full["Outlier"])
plt.title("Detection outliers using MyIF")
plt.xlabel("x1")
plt.ylabel("x2");


قد يلاحظ القارئ أن غابتنا تعمل لفترة كافية. ماذا سيحدث لمجموعة البيانات الأكبر؟ لكن لحسن الحظ، خلال التجارب، وجد مؤلفو هذه الخوارزمية أنه لا ينبغي استخدام جميع البيانات لبناء شجرة واحدة من الغابة، الأمر الذي لا يؤدي إلى زيادة سرعة العمل فحسب، بل يؤدي أيضًا إلى تحسين جودة اكتشاف الحالات الشاذة. وذلك لأن العينات الفرعية تحتوي على عدد أقل من النقاط العادية "التي تتداخل" مع الحالات الشاذة، مما يسهل عزل الحالات الشاذة. يظهر على الصورة أدناه.


In [ ]:
X_sample = X_full.sample(256)
plt.figure(figsize=(8, 8))
plt.scatter(
    X_sample["x1"], X_sample["x2"], c=X_sample["type"].map({"normal": 1, "anomaly": -1})
)
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("2d distribution of Sample (size=256)");


لن نقوم بتطوير IsolationForest الخاص بنا وسنستخدم إصدار sklearn مع هذه التحسينات بشكل أكبر.



# Sklearn هو كل شيء لدينا!


In [ ]:
# Import IsolationForest

from sklearn.ensemble import IsolationForest


#### معلمات IsolationForest المهمة:n_estimators - عدد المقدرين الأساسيين في المجموعة، الافتراضي=100.
    max_sample - عدد العينات من البيانات لتدريب كل شجرة في الغابة، الافتراضي "تلقائي" = min(256, n_samples)
    max_features - عدد الميزات من البيانات لتدريب كل شجرة في الغابة، الافتراضي = 1.0 (جميع الميزات)
    bootstrap - bootstrap، default=False
    التلوث - نسبة القيم المتطرفة في مجموعة البيانات، العتبة، الافتراضية = 0.1



سنختبر تنفيذ IF هذا على مجموعة البيانات ثنائية الأبعاد الخاصة بنا.


In [ ]:
isof = IsolationForest(random_state=77, n_jobs=4, contamination=0.05)

In [ ]:
%%time
isof.fit(X_full[["x1", "x2"]])


سريع جدا! ماذا عن الجودة ؟


In [ ]:
# predictions
y_pred_full = isof.predict(X_full[["x1", "x2"]])

In [ ]:
X_full["Outlier_sk"] = y_pred_full

In [ ]:
plt.figure(figsize=(10, 7))
plt.scatter(X_full["x1"], X_full["x2"], c=X_full["Outlier_sk"])
plt.title("Detection outliers using sklearn.IF")
plt.xlabel("x1")
plt.ylabel("x2");


الصورة تبدو صر! النقاط الأرجوانية هي القيم المتطرفة التي وجدتها IsolationForest. 


In [ ]:
# count of detected outliers "-1"
X_full.iloc[2000:, 4].value_counts()


كما يمكننا أن نرى، وجدت IsolationForest لدينا تقريبًا القيم المتطرفة.



# حان وقت التحدي!



في هذا الجزء من البرنامج التعليمي سنقوم بمقارنة IsolationForest مع خوارزميات أخرى. سوف نستخدم مجموعة البيانات هذه من Kaggle: https://www.kaggle.com/mlg-ulb/creditcardfraud


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from xgboost import XGBClassifier

pd.options.display.max_columns = 500

In [ ]:
data = pd.read_csv("creditcard.csv")


دعونا ننظر إلى البيانات. جميع الميزات رقمية، لذلك لن نقوم بأي معالجة للبيانات وسنقوم بتدريب النماذج على ما قمنا بتنزيله. لمنع الإفراط في تجهيز النماذج الخاضعة للإشراف، سنقوم بتقسيم مجموعة البيانات الأولية عن طريق التدريب وأجزاء الاختبار.


In [ ]:
data.head()

In [ ]:
data.describe()

In [ ]:
X = data.drop(columns=["Class"])
y = data["Class"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=17
)

In [ ]:
# Look at portion of fraud transactions
y.mean(), y_train.mean(), y_test.mean(), len(y[y == 1]), len(y), len(y_test), len(
    y_test[y_test == 1]
)


قف! تشكل معاملات الاحتيال حوالي 0.2٪ من جميع المعاملات!
سوف نستخدم جميع الخوارزميات "من الصندوق" دون ضبط المعلمات. نحن بحاجة إلى العثور على التحويلات الاحتيالية بسرعة وبقدر الإمكان.


In [ ]:
# supervised
rf = RandomForestClassifier(random_state=42, n_jobs=4)
lr = LogisticRegression(random_state=42)
xg = XGBClassifier(random_state=42, n_jobs=4)
# unsupervised
IF = IsolationForest(random_state=42, n_jobs=4, contamination=0.01, n_estimators=300)
LOF = LocalOutlierFactor(contamination=0.01, n_jobs=4)

In [ ]:
%%time
rf.fit(X_train, y_train)

In [ ]:
%%time
lr.fit(X_train, y_train)

In [ ]:
%%time
xg.fit(X_train, y_train)

In [ ]:
%%time
IF.fit(X_train, y_train)

In [ ]:
%%time
LOF.fit(X_train, y_train)

In [ ]:
rf_pred = rf.predict_proba(X_test)[:, 1]
lr_pred = lr.predict_proba(X_test)[:, 1]
xg_pred = xg.predict_proba(X_test)[:, 1]
IF_pred = IF.predict(X_test)
LOF_pred = LOF.fit_predict(X_test)

In [ ]:
X_test["true"] = y_test
X_test["IF_pred"] = IF_pred
X_test["LOF_pred"] = LOF_pred
X_test["xg_pred"] = xg_pred
X_test["rf_pred"] = rf_pred
X_test["lr_pred"] = lr_pred


دعونا نتحقق من عدد معاملات الاحتيال التي وجدتها كل خوارزمية:


In [ ]:
X_test[X_test["IF_pred"] == -1]["true"].value_counts()

In [ ]:
X_test[X_test["LOF_pred"] == -1]["true"].value_counts()

In [ ]:
X_test.sort_values(by="rf_pred", ascending=False)["true"].head(855).value_counts()

In [ ]:
X_test.sort_values(by="lr_pred", ascending=False)["true"].head(855).value_counts()

In [ ]:
X_test.sort_values(by="xg_pred", ascending=False)["true"].head(855).value_counts()


كما نرى، قامت Isolation Forest بحل هذه المشكلة بشكل أسوأ من الخوارزميات الخاضعة للإشراف، ولكنها بشكل عام جيدة جدًا للعشوائية. وبالمقارنة مع LOF، فإن النتيجة أفضل بكثير. وفي هذه الحالة لا نحتاج إلى معرفة المعاملات التي تعتبر احتيالًا - قم بإنشاء مجموعة بيانات قطار تحتوي على المتغير المستهدف.



#الخلاصة


في ختام البرنامج التعليمي، أريد تلخيص كل شيء يتعلق بـ Isolation Forest. تتمتع هذه الخوارزمية بفكرة بسيطة جدًا للكشف عن الحالات الشاذة: فهي تعزل الحالات الشاذة بدلاً من الملاحظات العادية. كما تتقارب Isolation Forest بسرعة مع حجم مجموعة صغير، مما يمكنها من اكتشاف الحالات الشاذة بكفاءة عالية.
إذا كنت تريد معرفة المزيد عن Isolation Forest، أنصحك بقراءة المقالة من مؤلفي الخوارزمية:
http://cs.nju.edu.cn/zhouzh/zhouzh.files/publication/icdm08b.pdf
https://cs.nju.edu.cn/zhouzh/zhouzh.files/publication/tkdd11.pdf